# SQL資料庫助手

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例  

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY
)

## LM Studio 或 Ollama
不建議使用太小的模型，經過測試gemma-3-12b-it產生的SQL指令不正確

In [ ]:
# from langchain_openai import ChatOpenAI
# model_name = 'gemma-3-12b-it'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# # 根據使用LM Studio或Ollama來選擇適當的伺服器URL
# base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
## base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

# llm = ChatOpenAI(
#     model=model_name,
#     openai_api_key="not-needed",
#     openai_api_base=base_url 
# )

## 測試模型

In [2]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("簡短的說明機器學習的定義")
]
result = llm.invoke(messages)
print(result.content)

機器學習是一種讓電腦**從數據中學習，而無需明確編程**的方法。

簡單來說，就是給電腦看很多例子，讓它自己找出規律和模式，然後用這些規律來預測或做出決策。

例如：

*   **垃圾郵件過濾:** 機器學習可以分析大量郵件，學習區分垃圾郵件和正常郵件的特徵。
*   **圖像識別:**  機器學習可以學習辨識圖片中的物體，例如貓、狗等。

總之，機器學習讓電腦擁有“學習”的能力，而不是單純地執行預先設定好的指令。



# Tools 對話機器人

## 準備工具

In [2]:
from langchain_core.tools import tool
import sqlite3

database_path = "sample_sales.db"
@tool 
def sql_query(sql: str):
    """
    執行 SQL 查詢並回傳結果。
    資料表結構可以利用 get_schema() 工具獲取。
    
    Args:
        sql: 完整的 SQLite 查詢語句。
        
    Returns:
        str: 查詢結果字串，每列資料以換行分隔，欄位間以逗號分隔。
    """    
    try:
        if not sql.strip().upper().startswith("SELECT"):
            return "錯誤：僅允許執行 SELECT 查詢。"

        conn = sqlite3.connect(database_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        rows = cursor.fetchall()
        result_lines = [", ".join(map(str, row)) for row in rows]
        result_str = "\n".join(result_lines)
        conn.close()
        return result_str
    except sqlite3.Error as e:
        return f"SQL 執行錯誤: {e}"


@tool 
def get_schema():
    """
    獲取資料庫中所有資料表的 Schema 資訊，包含資料表名稱與欄位定義。
    在撰寫 SQL 查詢前，應先呼叫此工具以了解資料庫結構。
    
    Returns:
        str: 包含所有 CREATE TABLE 語句的字串。
    """
    try:    
        conn = sqlite3.connect(database_path)
        cursor = conn.cursor()
        cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' ORDER BY name;")
        schemas = cursor.fetchall()
        schema_text = "\n\n".join([row[0] for row in schemas if row[0] is not None])
        conn.close()
        return schema_text       
    except sqlite3.Error as e:
        return f"SQL 執行錯誤: {e}"

tools_list = [sql_query, get_schema]

# 對話機器人

In [3]:
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage, AIMessageChunk
from langchain_core.output_parsers import StrOutputParser

class stream_chat_bot:
    def __init__(self, llm, tools):
        # 初始化對話機器人，傳入 LLM 與可用工具列表
        self.tools = tools
        # 將 LLM 綁定（bind）工具，使其具備自動呼叫工具的能力
        self.llm_with_tools = llm.bind_tools(tools)

        # 系統提示詞（System Prompt），用來設定 LLM 的角色與行為
        system_prompt = '''
你是一位智慧型個人助理，能夠根據使用者的問題主動判斷是否需要使用工具。
請以清楚、簡潔的方式回答問題。
若問題需要外部資料，請直接使用可用的工具完成查詢，不需向使用者確認。
'''        
        # 初始化訊息列表，第一條訊息是系統指令
        self.message = [SystemMessage(system_prompt)]

        # 將 LLM 的回應解析為純文字格式的工具
        self.str_parser = StrOutputParser()
       
    def chat_generator(self, text):
        """
        主對話生成函式（生成器形式）。
        逐步執行 LLM 回應與工具調用，並即時回傳每一步的結果。
        """
        # 將使用者的輸入加入訊息列表
        self.message.append(HumanMessage(text))        
        
        while True:
            # 呼叫 LLM，傳入完整訊息歷史
            
            final_ai_message = AIMessageChunk(content="")
            for chunk in self.llm_with_tools.stream(self.message):
                final_ai_message += chunk
                if hasattr(chunk, 'content') and chunk.content:
                    yield self.str_parser.invoke(chunk)
            
            response = final_ai_message
            
            # 將 LLM 回應加入訊息列表
            self.message.append(response)

            # 檢查 LLM 是否要求呼叫工具
            is_tools_call = False
            for tool_call in response.tool_calls:
                is_tools_call = True

                # 顯示 LLM 要執行的工具名稱與參數
                msg = f'[執行]: {tool_call["name"]}({tool_call["args"]})\n-----------\n' #完整訊息
                # msg = f'[執行]: {tool_call["name"]}()\n\n' #簡易訊息
                yield msg  # 使用 yield 讓結果能即時顯示在輸出中

                # 實際執行工具（根據工具名稱動態呼叫對應物件）
                tool_result = globals()[tool_call['name']].invoke(tool_call['args']) 

                # 顯示工具執行結果
                msg = f'[結果]: {tool_result}\n-----------\n'
                yield msg

                # 將工具執行結果封裝成 ToolMessage 回傳給 LLM
                tool_message = ToolMessage(
                    content=str(tool_result),          # 工具執行的文字結果
                    name=tool_call["name"],            # 工具名稱
                    tool_call_id=tool_call["id"],      # 工具呼叫 ID（讓 LLM 知道對應哪個呼叫）
                )
                # 將工具回傳結果加入訊息列表，提供 LLM 下一輪參考
                self.message.append(tool_message)
            
            # 若這一輪沒有任何工具呼叫，表示 LLM 已經生成最終回覆
            if is_tools_call == False:
                # 將 LLM 回應解析成純文字並輸出
                # yield self.str_parser.invoke(response)
                return  # 結束對話流程

    def chat(self, text, print_output=False):
        """
        封裝版對話函式。
        會收集 chat_generator 的所有輸出，並組合成完整的回覆字串。
        """
        msg = ''
        # 逐步取得 chat_generator 的產出內容
        for chunk in self.chat_generator(text):
            msg += f"{chunk}"
            if print_output:
                print(chunk, end='')
        # 回傳最終組合的對話內容
        return msg


In [4]:
bot = stream_chat_bot(llm, tools_list)

In [5]:
for x in bot.chat_generator("銷售數量最多的前五個商品？"):
    print(x, end='')

[執行]: get_schema({})
-----------
[結果]: CREATE TABLE Customers (
    CustomerID INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    BirthDate DATE,
    RegionID INTEGER,
    OccupationID INTEGER,
    FOREIGN KEY (RegionID) REFERENCES Regions(RegionID),
    FOREIGN KEY (OccupationID) REFERENCES Occupations(OccupationID)
)

CREATE TABLE Occupations (
    OccupationID INTEGER PRIMARY KEY,
    OccupationName TEXT NOT NULL UNIQUE
)

CREATE TABLE Products (
    ProductID INTEGER PRIMARY KEY,
    ProductName TEXT NOT NULL,
    Category TEXT,
    UnitPrice REAL NOT NULL
)

CREATE TABLE Regions (
    RegionID INTEGER PRIMARY KEY,
    RegionName TEXT NOT NULL UNIQUE
)

CREATE TABLE SaleItems (
    SaleID INTEGER,
    ProductID INTEGER,
    Quantity INTEGER NOT NULL,
    UnitPrice REAL NOT NULL,
    PRIMARY KEY (SaleID, ProductID),
    FOREIGN KEY (SaleID) REFERENCES Sales(SaleID),
    FOREIGN KEY (ProductID) REFERENCES Products(ProductID)
)

CREATE TABLE Sales (
    SaleID INTEGER PRIMARY KEY,
    C

In [6]:
for x in bot.chat_generator("2025年12月份購買總金額最高的前五名顧客？"):
    print(x, end='')

[執行]: get_schema({})
-----------
[結果]: CREATE TABLE Customers (
    CustomerID INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    BirthDate DATE,
    RegionID INTEGER,
    OccupationID INTEGER,
    FOREIGN KEY (RegionID) REFERENCES Regions(RegionID),
    FOREIGN KEY (OccupationID) REFERENCES Occupations(OccupationID)
)

CREATE TABLE Occupations (
    OccupationID INTEGER PRIMARY KEY,
    OccupationName TEXT NOT NULL UNIQUE
)

CREATE TABLE Products (
    ProductID INTEGER PRIMARY KEY,
    ProductName TEXT NOT NULL,
    Category TEXT,
    UnitPrice REAL NOT NULL
)

CREATE TABLE Regions (
    RegionID INTEGER PRIMARY KEY,
    RegionName TEXT NOT NULL UNIQUE
)

CREATE TABLE SaleItems (
    SaleID INTEGER,
    ProductID INTEGER,
    Quantity INTEGER NOT NULL,
    UnitPrice REAL NOT NULL,
    PRIMARY KEY (SaleID, ProductID),
    FOREIGN KEY (SaleID) REFERENCES Sales(SaleID),
    FOREIGN KEY (ProductID) REFERENCES Products(ProductID)
)

CREATE TABLE Sales (
    SaleID INTEGER PRIMARY KEY,
    C

In [35]:
for x in bot.chat_generator("第一名顧客該月的訂單明細？依照訂單日期分組"):
    print(x, end='')

[執行]: sql_query({'sql': "SELECT C.CustomerID FROM Customers AS C WHERE C.Name = '顧客49'"})
-----------
[結果]: 49
-----------
[執行]: sql_query({'sql': "SELECT S.SaleDate, P.ProductName, SI.Quantity, SI.UnitPrice, (SI.Quantity * SI.UnitPrice) AS ItemTotal FROM Sales AS S JOIN SaleItems AS SI ON S.SaleID = SI.SaleID JOIN Products AS P ON SI.ProductID = P.ProductID WHERE S.CustomerID = 49 AND S.SaleDate BETWEEN '2025-12-01 00:00:00' AND '2025-12-31 23:59:59' ORDER BY S.SaleDate"})
-----------
[結果]: 2025-12-01 04:46:00, 辦公桌, 3, 5000.0, 15000.0
2025-12-01 04:46:00, 運動鞋, 1, 2500.0, 2500.0
2025-12-14 20:20:00, 背包, 2, 2000.0, 4000.0
2025-12-14 20:20:00, 外套, 3, 3000.0, 9000.0
2025-12-14 20:20:00, 太陽眼鏡, 3, 1200.0, 3600.0
2025-12-14 20:20:00, 咖啡機, 3, 3500.0, 10500.0
2025-12-22 14:30:00, 行動電源, 2, 1200.0, 2400.0
2025-12-23 13:32:00, 藍牙音箱, 1, 2500.0, 2500.0
2025-12-28 04:26:00, 冰箱, 2, 22000.0, 44000.0
2025-12-28 04:26:00, 咖啡機, 3, 3500.0, 10500.0
2025-12-28 04:26:00, 太陽眼鏡, 1, 1200.0, 1200.0
2025-12-28 04